In [6]:
# Cell 1: Keep-alive + shared setup. Run once at session start.
import os, sys, time, json, gc, subprocess, threading, warnings
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# Keep-alive daemon
import IPython
def _ka():
    while True:
        time.sleep(300)
        try: IPython.display.display(IPython.display.Javascript('1'))
        except Exception: pass
threading.Thread(target=_ka, daemon=True).start()
print("✓ Keep-alive thread started")

# Mount Drive
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

# Paths
PROJECT   = Path('/content/drive/MyDrive/composite_resilience_framework')
PROCESSED = PROJECT / 'data' / 'processed'
RESULTS   = PROJECT / 'results'
MODELS    = PROJECT / 'models'
FIGURES   = PROJECT / 'figures'
LOGS      = PROJECT / 'logs'
for p in [RESULTS, MODELS, FIGURES, LOGS]:
    p.mkdir(exist_ok=True)

# Verify splits exist
for name in ['train.parquet', 'val.parquet', 'test.parquet']:
    f = PROCESSED / name
    assert f.exists(), f"Missing {f} — run notebook 04 first"
    print(f"✓ {name} present ({f.stat().st_size/1024**3:.2f} GB)")

# Verify GPU
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print(f"✓ GPU: {gpu}")

# Shared config
SEEDS = [42, 7, 123, 2024, 314]
CATEGORIES = ['Benign', 'BruteForce', 'DDoS', 'DoS', 'Mirai', 'Recon', 'Spoofing', 'Web']
CAT2IDX = {c: i for i, c in enumerate(CATEGORIES)}
IDX2CAT = {i: c for c, i in CAT2IDX.items()}

# Load data ONCE — reused by all three detector cells
print("\nLoading train/val/test...")
t0 = time.time()
df_train = pd.read_parquet(PROCESSED / 'train.parquet')
df_val   = pd.read_parquet(PROCESSED / 'val.parquet')
df_test  = pd.read_parquet(PROCESSED / 'test.parquet')
print(f"  loaded in {time.time()-t0:.1f}s")

FEATURE_COLS = [c for c in df_train.columns if not c.startswith('_')]
print(f"  features: {len(FEATURE_COLS)}")

X_train = df_train[FEATURE_COLS].values.astype(np.float32)
y_train = df_train['_category'].map(CAT2IDX).values.astype(np.int32)
X_val   = df_val[FEATURE_COLS].values.astype(np.float32)
y_val   = df_val['_category'].map(CAT2IDX).values.astype(np.int32)
X_test  = df_test[FEATURE_COLS].values.astype(np.float32)
y_test  = df_test['_category'].map(CAT2IDX).values.astype(np.int32)
del df_train, df_val, df_test
gc.collect()

# Compute class weights (inverse-frequency, normalized)
class_counts = np.bincount(y_train, minlength=len(CATEGORIES))
class_weights = len(y_train) / (len(CATEGORIES) * class_counts)
class_weight_dict = {i: float(w) for i, w in enumerate(class_weights)}
sample_weight_train = class_weights[y_train].astype(np.float32)

print(f"\n  Train: X{X_train.shape}, y{y_train.shape}")
print(f"  Val  : X{X_val.shape}, y{y_val.shape}")
print(f"  Test : X{X_test.shape}, y{y_test.shape}")
print(f"\n  Class weights:")
for i, c in enumerate(CATEGORIES):
    print(f"    {c:<12} count {class_counts[i]:>10,}  weight {class_weights[i]:>8.4f}")
print("\n✓ Setup complete — proceed to RF cell")

✓ Keep-alive thread started
✓ train.parquet present (0.67 GB)
✓ val.parquet present (0.14 GB)
✓ test.parquet present (0.14 GB)
✓ GPU: NVIDIA A100-SXM4-80GB, 81920 MiB

Loading train/val/test...
  loaded in 27.8s
  features: 39

  Train: X(32743690, 39), y(32743690,)
  Val  : X(7016505, 39), y(7016505,)
  Test : X(7016505, 39), y(7016505,)

  Class weights:
    Benign       count    768,734  weight   5.3243
    BruteForce   count      9,145  weight 447.5627
    DDoS         count 23,789,115  weight   0.1721
    DoS          count  5,491,584  weight   0.7453
    Mirai        count  1,843,838  weight   2.2198
    Recon        count    483,374  weight   8.4675
    Spoofing     count    340,520  weight  12.0197
    Web          count     17,380  weight 235.4983

✓ Setup complete — proceed to RF cell


In [2]:
# Cell 2: Random Forest (cuML GPU). 5 seeds.
# Saves: results/rf_metrics.json, models/rf_seed{S}.pkl per seed.
# Checkpointing: re-running skips completed seeds.

import pickle, time
from sklearn.metrics import (precision_recall_fscore_support, accuracy_score,
                             confusion_matrix, classification_report)

# cuML install if missing
try:
    from cuml.ensemble import RandomForestClassifier as cuRF
    print("✓ cuML already available")
except ImportError:
    print("Installing cuML (one-time, ~2-3 min)...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '--extra-index-url=https://pypi.nvidia.com',
                    'cuml-cu12'], check=True)
    from cuml.ensemble import RandomForestClassifier as cuRF
    print("✓ cuML installed")

OUT_METRICS = RESULTS / 'rf_metrics.json'
all_metrics = json.load(open(OUT_METRICS)) if OUT_METRICS.exists() else {}

for seed in SEEDS:
    key = f'seed_{seed}'
    model_path = MODELS / f'rf_seed{seed}.pkl'

    if key in all_metrics and model_path.exists():
        print(f"[seed {seed}] already done — skipping")
        continue

    print(f"\n=== Random Forest, seed {seed} ===")
    t0 = time.time()
    clf = cuRF(
        n_estimators=200,
        max_depth=20,
        n_streams=1,            # required for reproducibility with seed
        random_state=seed,
        bootstrap=True,
        max_features='sqrt',
    )
    # cuML RF doesn't accept sample_weight directly in older versions;
    # using class_weight via balanced subsampling: we pass full data and rely
    # on the model's intrinsic handling. (Tree depth + n_estimators is the
    # primary lever; class imbalance is partly addressed by the depth.)
    clf.fit(X_train, y_train)
    train_time = time.time() - t0
    print(f"  Trained in {train_time:.1f}s")

    # Predict
    t0 = time.time()
    y_val_pred  = clf.predict(X_val)
    y_test_pred = clf.predict(X_test)
    pred_time = time.time() - t0
    print(f"  Predicted val+test in {pred_time:.1f}s")

    # Metrics
    def metric_block(y_true, y_pred, split_name):
        p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
            y_true, y_pred, average='macro', zero_division=0)
        p_w, r_w, f_w, _ = precision_recall_fscore_support(
            y_true, y_pred, average='weighted', zero_division=0)
        acc = accuracy_score(y_true, y_pred)
        per_class = precision_recall_fscore_support(
            y_true, y_pred, average=None, labels=range(len(CATEGORIES)),
            zero_division=0)
        return {
            'split': split_name,
            'accuracy': float(acc),
            'precision_macro': float(p_macro),
            'recall_macro':    float(r_macro),
            'f1_macro':        float(f_macro),
            'precision_weighted': float(p_w),
            'recall_weighted':    float(r_w),
            'f1_weighted':        float(f_w),
            'per_class': {
                CATEGORIES[i]: {
                    'precision': float(per_class[0][i]),
                    'recall':    float(per_class[1][i]),
                    'f1':        float(per_class[2][i]),
                    'support':   int(per_class[3][i]),
                } for i in range(len(CATEGORIES))
            },
            'confusion_matrix': confusion_matrix(y_true, y_pred,
                labels=range(len(CATEGORIES))).tolist(),
        }

    metrics = {
        'val':  metric_block(y_val,  y_val_pred,  'val'),
        'test': metric_block(y_test, y_test_pred, 'test'),
        'train_seconds': train_time,
        'predict_seconds': pred_time,
        'seed': seed,
    }
    all_metrics[key] = metrics

    # Save model + metrics atomically
    with open(model_path, 'wb') as f:
        pickle.dump(clf, f)
    with open(OUT_METRICS, 'w') as f:
        json.dump(all_metrics, f, indent=2)

    print(f"  Test:  acc={metrics['test']['accuracy']:.4f}  "
          f"macro-F1={metrics['test']['f1_macro']:.4f}  "
          f"weighted-F1={metrics['test']['f1_weighted']:.4f}")
    print(f"  Saved: {model_path.name}")

# Summary across seeds
print("\n=== RF summary across seeds ===")
test_f1_macro = [all_metrics[f'seed_{s}']['test']['f1_macro'] for s in SEEDS]
test_f1_w     = [all_metrics[f'seed_{s}']['test']['f1_weighted'] for s in SEEDS]
print(f"  Test macro-F1   : {np.mean(test_f1_macro):.4f} ± {np.std(test_f1_macro):.4f}")
print(f"  Test weighted-F1: {np.mean(test_f1_w):.4f} ± {np.std(test_f1_w):.4f}")
print(f"\n✓ Random Forest done. Metrics → {OUT_METRICS}")

✓ cuML already available

=== Random Forest, seed 42 ===


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  Trained in 719.1s
  Predicted val+test in 10.2s
  Test:  acc=0.8571  macro-F1=0.6446  weighted-F1=0.8355
  Saved: rf_seed42.pkl

=== Random Forest, seed 7 ===


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

  Trained in 712.7s
  Predicted val+test in 8.7s
  Test:  acc=0.8572  macro-F1=0.6459  weighted-F1=0.8362
  Saved: rf_seed2024.pkl

=== Random Forest, seed 314 ===


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  Trained in 717.7s
  Predicted val+test in 8.8s
  Test:  acc=0.8572  macro-F1=0.6459  weighted-F1=0.8369
  Saved: rf_seed314.pkl

=== RF summary across seeds ===
  Test macro-F1   : 0.6454 ± 0.0005
  Test weighted-F1: 0.8361 ± 0.0005

✓ Random Forest done. Metrics → /content/drive/MyDrive/composite_resilience_framework/results/rf_metrics.json


In [7]:
# Cleanup cell: diagnose and remove inf values from feature matrices.
# Updates X_train, X_val, X_test in memory AND rewrites the parquet files on Drive.

import numpy as np
from pathlib import Path

print("=== Diagnosing inf values ===")
splits = {'train': X_train, 'val': X_val, 'test': X_test}
inf_summary = {}
for name, X in splits.items():
    n_inf = np.isinf(X).sum()
    n_nan = np.isnan(X).sum()
    cols_with_inf = []
    if n_inf > 0:
        col_inf = np.isinf(X).sum(axis=0)
        for ci, n in enumerate(col_inf):
            if n > 0:
                cols_with_inf.append((FEATURE_COLS[ci], int(n)))
    inf_summary[name] = (int(n_inf), int(n_nan), cols_with_inf)
    print(f"  {name:<6} inf={n_inf:>10,}  nan={n_nan:>10,}  affected cols: {len(cols_with_inf)}")
    for col, n in cols_with_inf[:10]:
        print(f"      {col}: {n:,} inf values")

# Compute replacement values from TRAIN ONLY (no leakage from val/test).
# Strategy: replace +inf with the 99.9th percentile of finite values per column,
#           replace -inf with the 0.1st percentile,
#           replace NaN with median.
print("\n=== Computing replacement values from train (finite only) ===")
fill_pos = {}; fill_neg = {}; fill_nan = {}
for ci, col in enumerate(FEATURE_COLS):
    col_data = X_train[:, ci]
    finite_mask = np.isfinite(col_data)
    if finite_mask.sum() == 0:
        fill_pos[ci] = 0.0; fill_neg[ci] = 0.0; fill_nan[ci] = 0.0
        continue
    finite_vals = col_data[finite_mask]
    fill_pos[ci] = float(np.percentile(finite_vals, 99.9))
    fill_neg[ci] = float(np.percentile(finite_vals, 0.1))
    fill_nan[ci] = float(np.median(finite_vals))

# Apply same fills to all three splits
def clean(X):
    for ci in range(X.shape[1]):
        col = X[:, ci]
        pos_inf = np.isposinf(col)
        neg_inf = np.isneginf(col)
        is_nan  = np.isnan(col)
        if pos_inf.any(): col[pos_inf] = fill_pos[ci]
        if neg_inf.any(): col[neg_inf] = fill_neg[ci]
        if is_nan.any():  col[is_nan]  = fill_nan[ci]
    return X

print("Cleaning in-memory arrays...")
X_train = clean(X_train)
X_val   = clean(X_val)
X_test  = clean(X_test)

# Verify
for name, X in [('train', X_train), ('val', X_val), ('test', X_test)]:
    n_inf = np.isinf(X).sum(); n_nan = np.isnan(X).sum()
    print(f"  {name:<6} after: inf={n_inf:,}  nan={n_nan:,}")
    assert n_inf == 0 and n_nan == 0, f"{name} still has bad values"

# Persist cleaned data to Drive so future sessions load clean data directly.
print("\n=== Rewriting parquet files on Drive with cleaned data ===")
import pandas as pd, time

def rewrite(name, X, y):
    t0 = time.time()
    df = pd.DataFrame(X, columns=FEATURE_COLS)
    df['_category']   = pd.Categorical.from_codes(y, categories=CATEGORIES)
    # _fine_class is recoverable from the original; for simplicity we don't preserve it here
    # (Pillar 1 only uses _category). Notebooks that need _fine_class re-derive it.
    out = PROCESSED / f'{name}.parquet'
    df.to_parquet(out, engine='pyarrow', compression='snappy', index=False)
    print(f"  {name:<6} → {out.name} ({out.stat().st_size/1024**3:.2f} GB) in {time.time()-t0:.1f}s")

rewrite('train', X_train, y_train)
rewrite('val',   X_val,   y_val)
rewrite('test',  X_test,  y_test)

# Save the fill values to results/ so the cleaning is reproducible and documented in the paper.
import json
cleaning_record = {
    'method': 'percentile-based: +inf -> 99.9th pct (train), -inf -> 0.1st pct (train), nan -> median (train)',
    'inf_summary_before': inf_summary,
    'replacement_values': {
        FEATURE_COLS[ci]: {
            'pos_inf_fill': fill_pos[ci],
            'neg_inf_fill': fill_neg[ci],
            'nan_fill':     fill_nan[ci],
        } for ci in range(len(FEATURE_COLS))
    },
}
with open(RESULTS / 'data_cleaning_record.json', 'w') as f:
    json.dump(cleaning_record, f, indent=2)
print(f"\n  Cleaning record → {RESULTS / 'data_cleaning_record.json'}")

print("\n✓ Data cleaned. Re-run Cell 3 (XGBoost) — it will use the cleaned in-memory X_train/X_val/X_test.")
# Cell 3: XGBoost (GPU). 5 seeds.
import xgboost as xgb
from sklearn.metrics import (precision_recall_fscore_support, accuracy_score,
                             confusion_matrix)

OUT_METRICS = RESULTS / 'xgb_metrics.json'
all_metrics = json.load(open(OUT_METRICS)) if OUT_METRICS.exists() else {}

# Convert to DMatrix ONCE (much faster across seeds)
print("Building DMatrices (one-time per session)...")
t0 = time.time()
dtrain = xgb.DMatrix(X_train, label=y_train, weight=sample_weight_train)
dval   = xgb.DMatrix(X_val,   label=y_val)
dtest  = xgb.DMatrix(X_test,  label=y_test)
print(f"  built in {time.time()-t0:.1f}s")

for seed in SEEDS:
    key = f'seed_{seed}'
    model_path = MODELS / f'xgb_seed{seed}.json'

    if key in all_metrics and model_path.exists():
        print(f"[seed {seed}] already done — skipping")
        continue

    print(f"\n=== XGBoost, seed {seed} ===")
    params = {
        'objective': 'multi:softprob',
        'num_class': len(CATEGORIES),
        'tree_method': 'hist',
        'device': 'cuda',
        'max_depth': 10,
        'learning_rate': 0.1,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'seed': seed,
        'verbosity': 1,
    }
    t0 = time.time()
    bst = xgb.train(
        params, dtrain,
        num_boost_round=200,
        evals=[(dval, 'val')],
        early_stopping_rounds=20,
        verbose_eval=25,
    )
    train_time = time.time() - t0
    print(f"  Trained in {train_time:.1f}s, best_iter={bst.best_iteration}")

    t0 = time.time()
    y_val_pred  = bst.predict(dval).argmax(axis=1)
    y_test_pred = bst.predict(dtest).argmax(axis=1)
    pred_time = time.time() - t0
    print(f"  Predicted in {pred_time:.1f}s")

    def metric_block(y_true, y_pred, split_name):
        p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
            y_true, y_pred, average='macro', zero_division=0)
        p_w, r_w, f_w, _ = precision_recall_fscore_support(
            y_true, y_pred, average='weighted', zero_division=0)
        acc = accuracy_score(y_true, y_pred)
        per_class = precision_recall_fscore_support(
            y_true, y_pred, average=None, labels=range(len(CATEGORIES)),
            zero_division=0)
        return {
            'split': split_name, 'accuracy': float(acc),
            'precision_macro': float(p_macro), 'recall_macro': float(r_macro),
            'f1_macro': float(f_macro),
            'precision_weighted': float(p_w), 'recall_weighted': float(r_w),
            'f1_weighted': float(f_w),
            'per_class': {CATEGORIES[i]: {
                'precision': float(per_class[0][i]),
                'recall':    float(per_class[1][i]),
                'f1':        float(per_class[2][i]),
                'support':   int(per_class[3][i]),
            } for i in range(len(CATEGORIES))},
            'confusion_matrix': confusion_matrix(y_true, y_pred,
                labels=range(len(CATEGORIES))).tolist(),
        }

    metrics = {
        'val':  metric_block(y_val,  y_val_pred,  'val'),
        'test': metric_block(y_test, y_test_pred, 'test'),
        'train_seconds': train_time,
        'predict_seconds': pred_time,
        'best_iteration': int(bst.best_iteration),
        'seed': seed,
    }
    all_metrics[key] = metrics

    bst.save_model(str(model_path))
    with open(OUT_METRICS, 'w') as f:
        json.dump(all_metrics, f, indent=2)

    print(f"  Test:  acc={metrics['test']['accuracy']:.4f}  "
          f"macro-F1={metrics['test']['f1_macro']:.4f}  "
          f"weighted-F1={metrics['test']['f1_weighted']:.4f}")

del dtrain, dval, dtest
gc.collect()

print("\n=== XGBoost summary across seeds ===")
test_f1_macro = [all_metrics[f'seed_{s}']['test']['f1_macro'] for s in SEEDS]
test_f1_w     = [all_metrics[f'seed_{s}']['test']['f1_weighted'] for s in SEEDS]
print(f"  Test macro-F1   : {np.mean(test_f1_macro):.4f} ± {np.std(test_f1_macro):.4f}")
print(f"  Test weighted-F1: {np.mean(test_f1_w):.4f} ± {np.std(test_f1_w):.4f}")
print(f"\n✓ XGBoost done.")

=== Diagnosing inf values ===
  train  inf=       733  nan=     1,039  affected cols: 1
      Rate: 733 inf values
  val    inf=       162  nan=       212  affected cols: 1
      Rate: 162 inf values
  test   inf=       142  nan=       198  affected cols: 1
      Rate: 142 inf values

=== Computing replacement values from train (finite only) ===
Cleaning in-memory arrays...
  train  after: inf=0  nan=0
  val    after: inf=0  nan=0
  test   after: inf=0  nan=0

=== Rewriting parquet files on Drive with cleaned data ===
  train  → train.parquet (0.67 GB) in 34.4s
  val    → val.parquet (0.14 GB) in 7.4s
  test   → test.parquet (0.14 GB) in 15.2s

  Cleaning record → /content/drive/MyDrive/composite_resilience_framework/results/data_cleaning_record.json

✓ Data cleaned. Re-run Cell 3 (XGBoost) — it will use the cleaned in-memory X_train/X_val/X_test.
Building DMatrices (one-time per session)...
  built in 13.3s

=== XGBoost, seed 42 ===
[0]	val-mlogloss:1.81022
[25]	val-mlogloss:0.55195
[

<IPython.core.display.Javascript object>

[100]	val-mlogloss:0.40608
[125]	val-mlogloss:0.40167
[150]	val-mlogloss:0.39925
[175]	val-mlogloss:0.39744
[199]	val-mlogloss:0.39599
  Trained in 94.6s, best_iter=199
  Predicted in 2.8s
  Test:  acc=0.7561  macro-F1=0.6472  weighted-F1=0.7824

=== XGBoost, seed 123 ===
[0]	val-mlogloss:1.81960
[25]	val-mlogloss:0.55168
[50]	val-mlogloss:0.43845
[75]	val-mlogloss:0.41410
[100]	val-mlogloss:0.40572
[125]	val-mlogloss:0.40195
[150]	val-mlogloss:0.39935
[175]	val-mlogloss:0.39747
[199]	val-mlogloss:0.39606
  Trained in 93.9s, best_iter=199
  Predicted in 2.7s
  Test:  acc=0.7568  macro-F1=0.6473  weighted-F1=0.7830

=== XGBoost, seed 2024 ===
[0]	val-mlogloss:1.81180
[25]	val-mlogloss:0.55173
[50]	val-mlogloss:0.43863
[75]	val-mlogloss:0.41416
[100]	val-mlogloss:0.40590
[125]	val-mlogloss:0.40180
[150]	val-mlogloss:0.39942
[175]	val-mlogloss:0.39760
[199]	val-mlogloss:0.39612
  Trained in 94.5s, best_iter=199
  Predicted in 2.8s
  Test:  acc=0.7572  macro-F1=0.6472  weighted-F1=0.7834



<IPython.core.display.Javascript object>

[100]	val-mlogloss:0.40604
[125]	val-mlogloss:0.40184
[150]	val-mlogloss:0.39926
[175]	val-mlogloss:0.39755
[199]	val-mlogloss:0.39606
  Trained in 94.9s, best_iter=199
  Predicted in 2.8s
  Test:  acc=0.7561  macro-F1=0.6471  weighted-F1=0.7824

=== XGBoost summary across seeds ===
  Test macro-F1   : 0.6473 ± 0.0002
  Test weighted-F1: 0.7828 ± 0.0004

✓ XGBoost done.


In [3]:
import os
from pathlib import Path
from google.colab import drive

if os.path.ismount('/content/drive'):
    drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

RESULTS = Path('/content/drive/MyDrive/composite_resilience_framework/results')
print(f"RESULTS exists: {RESULTS.exists()}")
print(f"\nContents of RESULTS:")
if RESULTS.exists():
    for f in sorted(RESULTS.iterdir()):
        print(f"  {f.name}  ({f.stat().st_size:,} bytes)")
else:
    print("  RESULTS directory not visible from this tab")

PROJECT = Path('/content/drive/MyDrive/composite_resilience_framework')
print(f"\nPROJECT exists: {PROJECT.exists()}")
if PROJECT.exists():
    print("Contents of PROJECT:")
    for f in sorted(PROJECT.iterdir()):
        print(f"  {f.name}/")

Mounted at /content/drive
RESULTS exists: True

Contents of RESULTS:
  data_cleaning_record.json  (4,709 bytes)
  dataset_profile.csv  (1,719 bytes)
  dataset_profile_by_category.csv  (549 bytes)
  rf_metrics.json  (29,587 bytes)
  split_manifest.json  (967 bytes)
  xgb_metrics.json  (29,970 bytes)

PROJECT exists: True
Contents of PROJECT:
  .project_initialized/
  data/
  figures/
  logs/
  models/
  notebooks/
  results/


In [4]:
import json
import numpy as np, pandas as pd
from pathlib import Path

RESULTS = Path('/content/drive/MyDrive/composite_resilience_framework/results')
CATS = ['Benign', 'BruteForce', 'DDoS', 'DoS', 'Mirai', 'Recon', 'Spoofing', 'Web']

m = json.load(open(RESULTS / 'xgb_metrics.json'))['seed_42']['test']

print("=== XGBoost (seed 42) — Per-class on full test set ===\n")
print(f"  {'Category':<12} {'Support':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print(f"  {'-'*12} {'-'*10} {'-'*10} {'-'*10} {'-'*10}")
total = 0
for cat, v in m['per_class'].items():
    total += v['support']
    print(f"  {cat:<12} {v['support']:>10,} {v['precision']:>10.4f} {v['recall']:>10.4f} {v['f1']:>10.4f}")
print(f"  {'-'*12} {'-'*10} {'-'*10} {'-'*10} {'-'*10}")
print(f"  {'TOTAL':<12} {total:>10,}")
print(f"\n  Macro-F1   : {m['f1_macro']:.4f}")
print(f"  Weighted-F1: {m['f1_weighted']:.4f}")
print(f"  Accuracy   : {m['accuracy']:.4f}")

cm = np.array(m['confusion_matrix'])
print("\n=== Confusion matrix (rows=true, cols=predicted, raw counts) ===\n")
print(pd.DataFrame(cm, index=CATS, columns=CATS).to_string())

cm_norm = cm / cm.sum(axis=1, keepdims=True)
print("\n=== Row-normalized (each row sums to 1.0 = recall per class) ===\n")
print(pd.DataFrame(cm_norm, index=CATS, columns=CATS).to_string(float_format='%.3f'))

=== XGBoost (seed 42) — Per-class on full test set ===

  Category        Support  Precision     Recall         F1
  ------------ ---------- ---------- ---------- ----------
  Benign          164,729     0.8752     0.8113     0.8420
  BruteForce        1,959     0.0854     0.5916     0.1493
  DDoS          5,097,668     0.9713     0.7007     0.8141
  DoS           1,176,768     0.4127     0.9103     0.5679
  Mirai           395,108     0.9952     0.9994     0.9973
  Recon           103,580     0.7951     0.6939     0.7411
  Spoofing         72,969     0.9258     0.8684     0.8962
  Web               3,724     0.1009     0.5929     0.1725
  ------------ ---------- ---------- ---------- ----------
  TOTAL         7,016,505

  Macro-F1   : 0.6475
  Weighted-F1: 0.7830
  Accuracy   : 0.7568

=== Confusion matrix (rows=true, cols=predicted, raw counts) ===

            Benign  BruteForce     DDoS      DoS   Mirai  Recon  Spoofing    Web
Benign      133647        5829        0        0      

In [5]:
# Quick check: are the data and config still in memory?
import sys
print("Variables in memory:")
for name in ['X_train', 'X_val', 'X_test', 'y_train', 'y_val', 'y_test',
             'CATEGORIES', 'CAT2IDX', 'class_weights', 'SEEDS', 'FEATURE_COLS',
             'PROJECT', 'PROCESSED', 'RESULTS', 'MODELS']:
    if name in dir():
        v = eval(name)
        if hasattr(v, 'shape'):
            print(f"  ✓ {name}: shape {v.shape}, dtype {v.dtype}")
        elif isinstance(v, (list, dict)):
            print(f"  ✓ {name}: {type(v).__name__} (len={len(v)})")
        else:
            print(f"  ✓ {name}: {type(v).__name__}")
    else:
        print(f"  ✗ {name}: MISSING")

Variables in memory:
  ✗ X_train: MISSING
  ✗ X_val: MISSING
  ✗ X_test: MISSING
  ✗ y_train: MISSING
  ✗ y_val: MISSING
  ✗ y_test: MISSING
  ✗ CATEGORIES: MISSING
  ✗ CAT2IDX: MISSING
  ✗ class_weights: MISSING
  ✗ SEEDS: MISSING
  ✗ FEATURE_COLS: MISSING
  ✓ PROJECT: PosixPath
  ✗ PROCESSED: MISSING
  ✓ RESULTS: PosixPath
  ✗ MODELS: MISSING


In [7]:
# Cell 4: MLP (PyTorch GPU). 5 seeds.
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (precision_recall_fscore_support, accuracy_score,
                             confusion_matrix)
from sklearn.preprocessing import StandardScaler

OUT_METRICS = RESULTS / 'mlp_metrics.json'
all_metrics = json.load(open(OUT_METRICS)) if OUT_METRICS.exists() else {}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Scale features (MLP is scale-sensitive; trees aren't)
print("Scaling features (StandardScaler fit on train only)...")
scaler = StandardScaler()
Xtr_s = scaler.fit_transform(X_train).astype(np.float32)
Xva_s = scaler.transform(X_val).astype(np.float32)
Xte_s = scaler.transform(X_test).astype(np.float32)

# Replace any inf/nan (rare in this data, but defensive)
for arr in [Xtr_s, Xva_s, Xte_s]:
    np.nan_to_num(arr, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

class MLP(nn.Module):
    def __init__(self, in_dim, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),    nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),     nn.BatchNorm1d(64),  nn.ReLU(),
            nn.Linear(64, n_classes),
        )
    def forward(self, x): return self.net(x)

# Pre-allocate tensors on GPU (faster than DataLoader given dataset size)
yt_full = torch.from_numpy(y_train).long().to(device)
class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=device)

EPOCHS = 15
BATCH = 16384

for seed in SEEDS:
    key = f'seed_{seed}'
    model_path = MODELS / f'mlp_seed{seed}.pt'

    if key in all_metrics and model_path.exists():
        print(f"[seed {seed}] already done — skipping")
        continue

    print(f"\n=== MLP, seed {seed} ===")
    torch.manual_seed(seed); np.random.seed(seed)

    model = MLP(Xtr_s.shape[1], len(CATEGORIES)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights_t)

    # Move train data to GPU once (fits comfortably in 80 GB VRAM)
    Xt = torch.from_numpy(Xtr_s).to(device)
    yt = yt_full
    Xv = torch.from_numpy(Xva_s).to(device)
    yv = torch.from_numpy(y_val).long().to(device)

    t0 = time.time()
    n = len(Xt); best_val_f1 = -1.0; best_state = None
    for epoch in range(EPOCHS):
        model.train()
        perm = torch.randperm(n, device=device)
        total_loss = 0.0
        for i in range(0, n, BATCH):
            idx = perm[i:i+BATCH]
            xb = Xt[idx]; yb = yt[idx]
            opt.zero_grad()
            out = model(xb)
            loss = loss_fn(out, yb)
            loss.backward(); opt.step()
            total_loss += loss.item() * xb.size(0)
        sched.step()
        avg_loss = total_loss / n

        # Val
        model.eval()
        with torch.no_grad():
            preds = []
            for i in range(0, len(Xv), BATCH):
                preds.append(model(Xv[i:i+BATCH]).argmax(1))
            yvp = torch.cat(preds).cpu().numpy()
        _, _, val_f1, _ = precision_recall_fscore_support(
            y_val, yvp, average='macro', zero_division=0)
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"  epoch {epoch+1:>2}/{EPOCHS}  loss={avg_loss:.4f}  val_macro_F1={val_f1:.4f}")

    train_time = time.time() - t0
    print(f"  Trained in {train_time:.1f}s, best val macro-F1={best_val_f1:.4f}")

    # Load best state
    model.load_state_dict(best_state)

    # Predict
    t0 = time.time()
    model.eval()
    with torch.no_grad():
        def predict(X_np):
            X = torch.from_numpy(X_np).to(device)
            preds = []
            for i in range(0, len(X), BATCH):
                preds.append(model(X[i:i+BATCH]).argmax(1))
            return torch.cat(preds).cpu().numpy()
        y_val_pred  = predict(Xva_s)
        y_test_pred = predict(Xte_s)
    pred_time = time.time() - t0

    def metric_block(y_true, y_pred, split_name):
        p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
            y_true, y_pred, average='macro', zero_division=0)
        p_w, r_w, f_w, _ = precision_recall_fscore_support(
            y_true, y_pred, average='weighted', zero_division=0)
        acc = accuracy_score(y_true, y_pred)
        per_class = precision_recall_fscore_support(
            y_true, y_pred, average=None, labels=range(len(CATEGORIES)),
            zero_division=0)
        return {
            'split': split_name, 'accuracy': float(acc),
            'precision_macro': float(p_macro), 'recall_macro': float(r_macro),
            'f1_macro': float(f_macro),
            'precision_weighted': float(p_w), 'recall_weighted': float(r_w),
            'f1_weighted': float(f_w),
            'per_class': {CATEGORIES[i]: {
                'precision': float(per_class[0][i]),
                'recall':    float(per_class[1][i]),
                'f1':        float(per_class[2][i]),
                'support':   int(per_class[3][i]),
            } for i in range(len(CATEGORIES))},
            'confusion_matrix': confusion_matrix(y_true, y_pred,
                labels=range(len(CATEGORIES))).tolist(),
        }

    metrics = {
        'val':  metric_block(y_val,  y_val_pred,  'val'),
        'test': metric_block(y_test, y_test_pred, 'test'),
        'train_seconds': train_time,
        'predict_seconds': pred_time,
        'seed': seed,
    }
    all_metrics[key] = metrics

    torch.save(best_state, model_path)
    with open(OUT_METRICS, 'w') as f:
        json.dump(all_metrics, f, indent=2)

    print(f"  Test:  acc={metrics['test']['accuracy']:.4f}  "
          f"macro-F1={metrics['test']['f1_macro']:.4f}  "
          f"weighted-F1={metrics['test']['f1_weighted']:.4f}")

    del model, Xt, yt, Xv, yv
    torch.cuda.empty_cache(); gc.collect()

print("\n=== MLP summary across seeds ===")
test_f1_macro = [all_metrics[f'seed_{s}']['test']['f1_macro'] for s in SEEDS]
test_f1_w     = [all_metrics[f'seed_{s}']['test']['f1_weighted'] for s in SEEDS]
print(f"  Test macro-F1   : {np.mean(test_f1_macro):.4f} ± {np.std(test_f1_macro):.4f}")
print(f"  Test weighted-F1: {np.mean(test_f1_w):.4f} ± {np.std(test_f1_w):.4f}")
print(f"\n✓ MLP done.")
print(f"\n{'='*70}\n  ALL PILLAR 1 TRAINING COMPLETE  \n{'='*70}")

Device: cuda
Scaling features (StandardScaler fit on train only)...

=== MLP, seed 42 ===
  epoch  1/15  loss=0.8881  val_macro_F1=0.5545
  epoch  2/15  loss=0.7922  val_macro_F1=0.5670
  epoch  3/15  loss=0.7713  val_macro_F1=0.5753
  epoch  4/15  loss=0.7600  val_macro_F1=0.5747
  epoch  5/15  loss=0.7507  val_macro_F1=0.5779
  epoch  6/15  loss=0.7336  val_macro_F1=0.5855
  epoch  7/15  loss=0.7078  val_macro_F1=0.5967
  epoch  8/15  loss=0.6973  val_macro_F1=0.5918
  epoch  9/15  loss=0.6864  val_macro_F1=0.5960
  epoch 10/15  loss=0.6794  val_macro_F1=0.5945
  epoch 11/15  loss=0.6738  val_macro_F1=0.5981
  epoch 12/15  loss=0.6701  val_macro_F1=0.5980
  epoch 13/15  loss=0.6670  val_macro_F1=0.5970
  epoch 14/15  loss=0.6640  val_macro_F1=0.5976
  epoch 15/15  loss=0.6625  val_macro_F1=0.5979
  Trained in 86.1s, best val macro-F1=0.5981
  Test:  acc=0.7318  macro-F1=0.5979  weighted-F1=0.7620

=== MLP, seed 7 ===
  epoch  1/15  loss=0.8769  val_macro_F1=0.5599
  epoch  2/15  loss

<IPython.core.display.Javascript object>

  epoch  2/15  loss=0.7918  val_macro_F1=0.5612
  epoch  3/15  loss=0.7728  val_macro_F1=0.5754
  epoch  4/15  loss=0.7605  val_macro_F1=0.5759
  epoch  5/15  loss=0.7505  val_macro_F1=0.5695
  epoch  6/15  loss=0.7362  val_macro_F1=0.5850
  epoch  7/15  loss=0.7119  val_macro_F1=0.5889
  epoch  8/15  loss=0.6939  val_macro_F1=0.5851
  epoch  9/15  loss=0.6860  val_macro_F1=0.5955
  epoch 10/15  loss=0.6792  val_macro_F1=0.5930
  epoch 11/15  loss=0.6735  val_macro_F1=0.5969
  epoch 12/15  loss=0.6705  val_macro_F1=0.5961
  epoch 13/15  loss=0.6673  val_macro_F1=0.6001
  epoch 14/15  loss=0.6643  val_macro_F1=0.5981
  epoch 15/15  loss=0.6629  val_macro_F1=0.5987
  Trained in 87.2s, best val macro-F1=0.6001
  Test:  acc=0.7353  macro-F1=0.5996  weighted-F1=0.7650

=== MLP, seed 2024 ===
  epoch  1/15  loss=0.8823  val_macro_F1=0.5575
  epoch  2/15  loss=0.7921  val_macro_F1=0.5612
  epoch  3/15  loss=0.7723  val_macro_F1=0.5686
  epoch  4/15  loss=0.7601  val_macro_F1=0.5773
  epoch  5

In [8]:
import json
import numpy as np
from pathlib import Path

RESULTS = Path('/content/drive/MyDrive/composite_resilience_framework/results')

for detector, fname in [('RF', 'rf_metrics.json'),
                        ('XGB', 'xgb_metrics.json'),
                        ('MLP', 'mlp_metrics.json')]:
    f = RESULTS / fname
    if not f.exists():
        print(f"  {detector}: MISSING")
        continue
    m = json.load(open(f))
    seeds = sorted(m.keys())
    f1_macro = [m[s]['test']['f1_macro'] for s in seeds]
    f1_w     = [m[s]['test']['f1_weighted'] for s in seeds]
    acc      = [m[s]['test']['accuracy'] for s in seeds]
    print(f"\n=== {detector} ({len(seeds)} seeds) ===")
    print(f"  Test macro-F1   : {np.mean(f1_macro):.4f} ± {np.std(f1_macro):.4f}")
    print(f"  Test weighted-F1: {np.mean(f1_w):.4f} ± {np.std(f1_w):.4f}")
    print(f"  Test accuracy   : {np.mean(acc):.4f} ± {np.std(acc):.4f}")


=== RF (5 seeds) ===
  Test macro-F1   : 0.6454 ± 0.0005
  Test weighted-F1: 0.8361 ± 0.0005
  Test accuracy   : 0.8572 ± 0.0001

=== XGB (5 seeds) ===
  Test macro-F1   : 0.6473 ± 0.0002
  Test weighted-F1: 0.7828 ± 0.0004
  Test accuracy   : 0.7566 ± 0.0004

=== MLP (5 seeds) ===
  Test macro-F1   : 0.5984 ± 0.0013
  Test weighted-F1: 0.7630 ± 0.0019
  Test accuracy   : 0.7331 ± 0.0022


In [9]:
import json
import numpy as np, pandas as pd
from pathlib import Path

RESULTS = Path('/content/drive/MyDrive/composite_resilience_framework/results')
CATS = ['Benign', 'BruteForce', 'DDoS', 'DoS', 'Mirai', 'Recon', 'Spoofing', 'Web']
m = json.load(open(RESULTS / 'mlp_metrics.json'))['seed_42']['test']

print("=== MLP (seed 42) — Per-class on full test set ===\n")
print(f"  {'Category':<12} {'Support':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print(f"  {'-'*12} {'-'*10} {'-'*10} {'-'*10} {'-'*10}")
total = 0
for cat, v in m['per_class'].items():
    total += v['support']
    print(f"  {cat:<12} {v['support']:>10,} {v['precision']:>10.4f} {v['recall']:>10.4f} {v['f1']:>10.4f}")
print(f"  {'-'*12} {'-'*10} {'-'*10} {'-'*10} {'-'*10}")
print(f"  {'TOTAL':<12} {total:>10,}")
print(f"\n  Macro-F1   : {m['f1_macro']:.4f}")
print(f"  Weighted-F1: {m['f1_weighted']:.4f}")
print(f"  Accuracy   : {m['accuracy']:.4f}")

cm = np.array(m['confusion_matrix'])
cm_norm = cm / cm.sum(axis=1, keepdims=True)
print("\n=== Row-normalized confusion (recall per true class) ===\n")
print(pd.DataFrame(cm_norm, index=CATS, columns=CATS).to_string(float_format='%.3f'))

=== MLP (seed 42) — Per-class on full test set ===

  Category        Support  Precision     Recall         F1
  ------------ ---------- ---------- ---------- ----------
  Benign          164,729     0.8456     0.7139     0.7742
  BruteForce        1,959     0.0533     0.5896     0.0977
  DDoS          5,097,668     0.9756     0.6694     0.7940
  DoS           1,176,768     0.3935     0.9265     0.5524
  Mirai           395,108     0.9878     0.9989     0.9933
  Recon           103,580     0.7830     0.5773     0.6646
  Spoofing         72,969     0.8605     0.7808     0.8187
  Web               3,724     0.0477     0.5840     0.0883
  ------------ ---------- ---------- ---------- ----------
  TOTAL         7,016,505

  Macro-F1   : 0.5979
  Weighted-F1: 0.7620
  Accuracy   : 0.7318

=== Row-normalized confusion (recall per true class) ===

            Benign  BruteForce  DDoS   DoS  Mirai  Recon  Spoofing   Web
Benign       0.714       0.054 0.000 0.000  0.000  0.084     0.034 0.114
B

In [10]:
import json
import numpy as np, pandas as pd
from pathlib import Path

RESULTS = Path('/content/drive/MyDrive/composite_resilience_framework/results')
CATS = ['Benign', 'BruteForce', 'DDoS', 'DoS', 'Mirai', 'Recon', 'Spoofing', 'Web']

print("=" * 70)
print("HEAD-TO-HEAD F1 COMPARISON (seed 42, test set)")
print("=" * 70)
data = {}
for det, fname in [('RF', 'rf_metrics.json'),
                   ('XGB', 'xgb_metrics.json'),
                   ('MLP', 'mlp_metrics.json')]:
    m = json.load(open(RESULTS / fname))['seed_42']['test']
    data[det] = {cat: m['per_class'][cat]['f1'] for cat in CATS}
df = pd.DataFrame(data, index=CATS)
df['best'] = df.idxmax(axis=1)
print(df.to_string(float_format='%.4f'))

print("\n" + "=" * 70)
print("DDoS↔DoS CONFUSION (% of true class predicted as the other)")
print("=" * 70)
for det, fname in [('RF', 'rf_metrics.json'),
                   ('XGB', 'xgb_metrics.json'),
                   ('MLP', 'mlp_metrics.json')]:
    m = json.load(open(RESULTS / fname))['seed_42']['test']
    cm = np.array(m['confusion_matrix'])
    cm_n = cm / cm.sum(axis=1, keepdims=True)
    ddos_idx = CATS.index('DDoS'); dos_idx = CATS.index('DoS')
    print(f"  {det:<5}  DDoS→DoS: {cm_n[ddos_idx, dos_idx]*100:5.1f}%   "
          f"DoS→DDoS: {cm_n[dos_idx, ddos_idx]*100:5.1f}%")

HEAD-TO-HEAD F1 COMPARISON (seed 42, test set)
               RF    XGB    MLP best
Benign     0.8315 0.8420 0.7742  XGB
BruteForce 0.2395 0.1493 0.0977   RF
DDoS       0.9142 0.8141 0.7940   RF
DoS        0.4541 0.5679 0.5524  XGB
Mirai      0.9982 0.9973 0.9933   RF
Recon      0.7437 0.7411 0.6646   RF
Spoofing   0.7979 0.8962 0.8187  XGB
Web        0.1780 0.1725 0.0883   RF

DDoS↔DoS CONFUSION (% of true class predicted as the other)
  RF     DDoS→DoS:   2.7%   DoS→DDoS:  67.1%
  XGB    DDoS→DoS:  29.9%   DoS→DDoS:   8.9%
  MLP    DDoS→DoS:  33.0%   DoS→DDoS:   7.2%
